# Book Recommendation Engine using KNN

Given a book title, return 5 similar books based on user rating patterns, using K-Nearest Neighbors on a book-by-user rating matrix (cosine distance).

Remove statistically insignificant data first: users with fewer than 200 ratings, books with fewer than 100 ratings.

## 1. *Libraries importation*

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import urllib.request
import zipfile


## 2. *Get data files*

In [2]:
# Download the zip file
headers = {"User-Agent": "Mozilla/5.0"}
url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"
zip_filename = "book-crossings.zip"

req = urllib.request.Request(url, headers=headers)
with urllib.request.urlopen(req) as response, open(zip_filename, "wb") as out_file:
    out_file.write(response.read())

# Unzip it
with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(".")  # extracts into the current folder

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

print("Files extracted:", os.listdir("."))

Files extracted: ['book-crossings.zip', 'BX-Book-Ratings.csv', 'BX-Books.csv', 'BX-Users.csv', 'fcc_book_recommendation_knn.ipynb', 'README.md']


## 3. *Load dataframes*

In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})


In [4]:
df_books.head()

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


## 4. *Filter sparse users and books*

In [8]:
# count the number of ratings per user and per book
user_counts = df_ratings['user'].value_counts()
book_counts = df_ratings['isbn'].value_counts()

# filter out users with less than 200 ratings and books with less than 100 ratings
active_users = user_counts[user_counts >= 200].index
popular_books = book_counts[book_counts >= 100].index

# filter the ratings dataframe
df_ratings_filtered = df_ratings[
    (df_ratings['user'].isin(active_users)) & (df_ratings['isbn'].isin(popular_books))
]

## 5. *Merge and pivot into a rating matrix*

In [12]:
# create a pivot table with books as rows, users as columns, and ratings as values
book_pivot = pd.merge(df_ratings_filtered, df_books, on='isbn').pivot_table(
        index='title', 
        columns='user', 
        values='rating'
    ).fillna(0)

book_pivot

user,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Beautiful Mind: The Life of Mathematical Genius and Nobel Laureate John Nash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Without Remorse,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,7.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 6. *Fit the Nearest Neighbors model*

In [13]:
# Convert the pivot table to a sparse matrix
sparse_matrix = csr_matrix(book_pivot.values)

# Create a NearestNeighbors model using cosine similarity
model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6, n_jobs=-1).fit(sparse_matrix)

## 7. *get_recommends function*

In [23]:
def get_recommends(book = ""):
    book_index = book_pivot.index.get_loc(book)
    distances, indices = model.kneighbors(sparse_matrix[book_index], n_neighbors=6)

    recommended_books = []
    for i in range(len(distances.flatten()) - 1, 0, -1):
        title = book_pivot.index[indices.flatten()[i]]
        dist = distances.flatten()[i]
        recommended_books.append([title, dist])

    return [book, recommended_books]

books = get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")  # Other ex to test with "Where the Heart Is (Oprah's Book Club (Paperback))"
books

['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['Catch 22', np.float32(0.7939835)],
  ['The Witching Hour (Lives of the Mayfair Witches)', np.float32(0.74486566)],
  ['Interview with the Vampire', np.float32(0.73450685)],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))',
   np.float32(0.53763384)],
  ['The Vampire Lestat (Vampire Chronicles, Book II)', np.float32(0.5178412)]]]

## 8. *Final test cell*

In [24]:
def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! \U0001F389\U0001F389\U0001F389\U0001F389\U0001F389")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()


You passed the challenge! 🎉🎉🎉🎉🎉
